# Notebook 10 — Horizon 5 Hari (Skripsi-Scoped Random Search, reuse `notebook07_common.py`)

**Beda dari notebook 03/06 (metodologi "v5")**: notebook ini TIDAK memakai indikator
teknikal, grid search, ensemble, atau walk-forward. Notebook ini **reuse langsung**
pipeline produksi yang dipakai model skripsi resmi (`07_LSTM_FinBERT_skripsi_scoped.ipynb`
dan endpoint API `training_service.py`):

- Loader data: `pipeline/feature_engineering/dataset_builder.py` (`load_joined()` /
  `load_ohlcv_only()`) — SAMA PERSIS, tidak reinvent.
- Logic training/eval: `models/lstm/notebook07_common.py::run_notebook07_training()` —
  Random Search 20 iterasi, `FlexibleLSTMClassifier`, capping outlier IQR (fit-train-only),
  split kronologis 70/15/15, evaluasi HANYA 4 metrik (Accuracy/Precision/Recall/F1) — SAMA
  PERSIS dengan yang menghasilkan model produksi horizon-1.
- Search space, jumlah iterasi, window: `models/lstm/notebook07_trainer.py::SEARCH_SPACE
  / N_ITER / WINDOW` — SAMA PERSIS, tidak diubah.

**Satu-satunya variabel yang berbeda dari model produksi: `HORIZON = 5`** (bukan 1).
Ini baru bisa dilakukan setelah `notebook07_common.py` diberi parameter `horizon`
(default tetap 1 — perilaku API/produksi TIDAK berubah; lihat docstring modul).

**Prinsip anti p-hacking (konsisten dengan seluruh notebook project ini):** horizon 1
(model resmi BAB I) dan horizon 5 (eksperimen ini) DILAPORKAN BERDAMPINGAN apa adanya
di Bagian 4 — bukan cuma yang "menang" yang ditampilkan. Horizon 1 tetap satu-satunya
hasil resmi skripsi; horizon 5 murni analisis sensitivitas/robustness di luar Batasan
Masalah BAB I.

Artefak disimpan dengan suffix `_h5` (`lstm_baseline_skripsi_h5.*` /
`finbert_lstm_skripsi_h5.*`) — TIDAK menimpa artefak horizon-1 produksi.

**Status: notebook ini ditulis LENGKAP tapi BELUM DIEKSEKUSI.** Semua cell `outputs=[]`.


## Bagian 1 — Data

Reuse `pipeline/feature_engineering/dataset_builder.py` (identik dengan notebook 07):
- `load_joined()` → OHLCV + sentimen harian, `FEATURE_COLS` — dipakai Model B (FinBERT-LSTM).
- `load_ohlcv_only()` → OHLCV saja, `OHLCV_FEATURE_COLS` — dipakai Model A (baseline).


In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False else Path.cwd()
if (PROJECT_ROOT / "pipeline").exists() is False:
    PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

import logging

import pandas as pd
import torch

from pipeline.feature_engineering.dataset_builder import (
    FEATURE_COLS,
    OHLCV_FEATURE_COLS,
    load_joined,
    load_ohlcv_only,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger = logging.getLogger("notebook10")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info("Device: %s", DEVICE)


2026-07-24 00:25:00,741 [INFO] notebook10: Device: cuda


In [2]:
df_ohlcv = load_ohlcv_only()   # Model A -- OHLCV_FEATURE_COLS
df_joined = load_joined()      # Model B -- FEATURE_COLS (OHLCV + sentiment_mean)

print("Model A (OHLCV-only)  --", df_ohlcv.shape, "kolom:", list(df_ohlcv.columns))
print("Model B (FinBERT-LSTM) --", df_joined.shape, "kolom:", list(df_joined.columns))
print("Rentang tanggal:", df_joined["Date"].min().date(), "s.d.", df_joined["Date"].max().date())


2026-07-24 00:25:00,766 [INFO] pipeline.feature_engineering.dataset_builder: OHLCV-only dataset: 1552 rows (2018-01-02 – 2024-03-04)
2026-07-24 00:25:00,781 [INFO] pipeline.feature_engineering.dataset_builder: Stock loaded: 1552 rows (2018-01-02 – 2024-03-04)
2026-07-24 00:25:00,782 [INFO] pipeline.feature_engineering.dataset_builder: Sentiment loaded: 3504 rows (2008-01-02 – 2024-03-04)
2026-07-24 00:25:00,786 [INFO] pipeline.feature_engineering.dataset_builder: Joined dataset: 1528 rows (2018-01-02 – 2024-03-04)


Model A (OHLCV-only)  -- (1552, 6) kolom: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
Model B (FinBERT-LSTM) -- (1528, 7) kolom: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'sentiment_mean']
Rentang tanggal: 2018-01-02 s.d. 2024-03-04


## Bagian 2 — Konfigurasi Horizon 5 (satu-satunya variabel yang berubah dari produksi)

`SEARCH_SPACE`, `N_ITER`, `WINDOW` diimpor APA ADANYA dari `notebook07_trainer.py` —
TIDAK disesuaikan/di-tuning ulang untuk horizon 5 (apples-to-apples, hindari p-hacking).
`ARTIFACT_DIR` dibangun manual (absolut via `PROJECT_ROOT`) karena notebook dijalankan
dari folder `notebooks/`, bukan dari root project seperti `training_service.py`.


In [3]:
from models.lstm.notebook07_common import load_split_capped, run_notebook07_training
from models.lstm.notebook07_trainer import (
    BASELINE_FILES,
    FINBERT_FILES,
    N_ITER,
    SEARCH_SPACE,
    WINDOW,
)

HORIZON = 5  # satu-satunya perbedaan dari model produksi (horizon=1)
ARTIFACT_DIR = str(PROJECT_ROOT / "data" / "artifacts" / "lstm_weights")

H5_BASELINE_FILES = {
    "model": "lstm_baseline_skripsi_h5.pt",
    "scaler": "lstm_baseline_skripsi_h5_scaler.joblib",
    "metadata": "lstm_baseline_skripsi_h5_metadata.json",
}
H5_FINBERT_FILES = {
    "model": "finbert_lstm_skripsi_h5.pt",
    "scaler": "finbert_lstm_skripsi_h5_scaler.joblib",
    "metadata": "finbert_lstm_skripsi_h5_metadata.json",
}


def make_emit(label: str):
    """Emit callback ringkas -- cetak hanya start/iteration_end/done/error
    (skip epoch-level, N_ITER=20 x <=80 epoch bisa ribuan baris kalau semua dicetak)."""

    def _emit(event: dict) -> None:
        etype = event.get("type")
        if etype == "start":
            print(f"[{label}] start | window={event['window']} horizon={event['horizon']} "
                  f"n_iter={event['total_iterations']} device={event['device']}")
            s = event["split"]
            print(f"[{label}] split rows  train={s['train_rows']} val={s['val_rows']} test={s['test_rows']}")
            print(f"[{label}] split samples train={s['train_samples']} val={s['val_samples']} test={s['test_samples']}")
        elif etype == "iteration_end":
            marker = "  <- NEW BEST" if event["is_new_best"] else ""
            print(f"[{label}] iter {event['iteration']:>2}/{event['total_iterations']} "
                  f"val_f1={event['val_f1']:.4f} val_acc={event['val_acc']:.4f}{marker}")
        elif etype == "done":
            print(f"[{label}] DONE | best_val_f1={event['best_val_f1']:.4f} best_val_acc={event['best_val_acc']:.4f}")
            print(f"[{label}] test metrics: {event['metrics']}")
            print(f"[{label}] confusion matrix: {event['confusion_matrix']}")
        elif etype == "error":
            print(f"[{label}] ERROR: {event['message']}")

    return _emit


## Bagian 2b — Cek Distribusi Label Horizon 5 (sebelum training)

Transparansi seperti notebook lain di project ini -- lihat keseimbangan kelas Up/Down
per split SEBELUM melatih model, supaya collapse (kalau terjadi) tidak mengejutkan di
Bagian 3.


In [4]:
for label, load_fn, cols in [
    ("Model A (OHLCV)", load_ohlcv_only, ["Volume"]),
    ("Model B (FinBERT-LSTM)", load_joined, ["Volume", "sentiment_mean"]),
]:
    df_tr, df_vl, df_te, _, _ = load_split_capped(load_fn, cols, horizon=HORIZON)
    for split_name, d in [("train", df_tr), ("val", df_vl), ("test", df_te)]:
        up_pct = d["label"].mean() * 100
        print(f"{label:24s} | {split_name:5s} | n={len(d):4d} | Up%={up_pct:5.1f}")


2026-07-24 00:25:00,842 [INFO] pipeline.feature_engineering.dataset_builder: OHLCV-only dataset: 1552 rows (2018-01-02 – 2024-03-04)
2026-07-24 00:25:00,859 [INFO] pipeline.feature_engineering.dataset_builder: Stock loaded: 1552 rows (2018-01-02 – 2024-03-04)
2026-07-24 00:25:00,860 [INFO] pipeline.feature_engineering.dataset_builder: Sentiment loaded: 3504 rows (2008-01-02 – 2024-03-04)
2026-07-24 00:25:00,864 [INFO] pipeline.feature_engineering.dataset_builder: Joined dataset: 1528 rows (2018-01-02 – 2024-03-04)


Model A (OHLCV)          | train | n=1082 | Up%= 61.6
Model A (OHLCV)          | val   | n= 232 | Up%= 50.9
Model A (OHLCV)          | test  | n= 233 | Up%= 65.2
Model B (FinBERT-LSTM)   | train | n=1066 | Up%= 61.7
Model B (FinBERT-LSTM)   | val   | n= 228 | Up%= 51.3
Model B (FinBERT-LSTM)   | test  | n= 229 | Up%= 65.5


## Bagian 3 — Random Search Horizon 5 (Model A & Model B)

Memanggil `run_notebook07_training()` LANGSUNG -- bukan re-implementasi -- dengan
parameter identik notebook 07 kecuali `horizon=5` dan nama artefak (`H5_*_FILES`).


In [5]:
result_a_h5 = run_notebook07_training(
    emit=make_emit("Model A (OHLCV) h5"),
    feature_cols=OHLCV_FEATURE_COLS,
    load_data_fn=load_ohlcv_only,
    capping_cols=["Volume"],
    search_space=SEARCH_SPACE,
    n_iter=N_ITER,
    window=WINDOW,
    artifact_dir=ARTIFACT_DIR,
    model_filename=H5_BASELINE_FILES["model"],
    scaler_filename=H5_BASELINE_FILES["scaler"],
    metadata_filename=H5_BASELINE_FILES["metadata"],
    horizon=HORIZON,
)


2026-07-24 00:25:00,883 [INFO] pipeline.feature_engineering.dataset_builder: OHLCV-only dataset: 1552 rows (2018-01-02 – 2024-03-04)


[Model A (OHLCV) h5] start | window=10 horizon=5 n_iter=20 device=cuda
[Model A (OHLCV) h5] split rows  train=1082 val=232 test=233
[Model A (OHLCV) h5] split samples train=1073 val=223 test=224
[Model A (OHLCV) h5] iter  1/20 val_f1=0.6844 val_acc=0.5202  <- NEW BEST
[Model A (OHLCV) h5] iter  2/20 val_f1=0.6844 val_acc=0.5202
[Model A (OHLCV) h5] iter  3/20 val_f1=0.6255 val_acc=0.5919
[Model A (OHLCV) h5] iter  4/20 val_f1=0.6844 val_acc=0.5202
[Model A (OHLCV) h5] iter  5/20 val_f1=0.6844 val_acc=0.5202
[Model A (OHLCV) h5] iter  6/20 val_f1=0.1884 val_acc=0.4978
[Model A (OHLCV) h5] iter  7/20 val_f1=0.0000 val_acc=0.4798
[Model A (OHLCV) h5] iter  8/20 val_f1=0.0000 val_acc=0.4753
[Model A (OHLCV) h5] iter  9/20 val_f1=0.6884 val_acc=0.5291  <- NEW BEST
[Model A (OHLCV) h5] iter 10/20 val_f1=0.6844 val_acc=0.5202
[Model A (OHLCV) h5] iter 11/20 val_f1=0.6844 val_acc=0.5202
[Model A (OHLCV) h5] iter 12/20 val_f1=0.7059 val_acc=0.5740  <- NEW BEST
[Model A (OHLCV) h5] iter 13/20 va

In [6]:
result_b_h5 = run_notebook07_training(
    emit=make_emit("Model B (FinBERT-LSTM) h5"),
    feature_cols=FEATURE_COLS,
    load_data_fn=load_joined,
    capping_cols=["Volume", "sentiment_mean"],
    search_space=SEARCH_SPACE,
    n_iter=N_ITER,
    window=WINDOW,
    artifact_dir=ARTIFACT_DIR,
    model_filename=H5_FINBERT_FILES["model"],
    scaler_filename=H5_FINBERT_FILES["scaler"],
    metadata_filename=H5_FINBERT_FILES["metadata"],
    horizon=HORIZON,
)


2026-07-24 00:26:49,584 [INFO] pipeline.feature_engineering.dataset_builder: Stock loaded: 1552 rows (2018-01-02 – 2024-03-04)
2026-07-24 00:26:49,585 [INFO] pipeline.feature_engineering.dataset_builder: Sentiment loaded: 3504 rows (2008-01-02 – 2024-03-04)
2026-07-24 00:26:49,591 [INFO] pipeline.feature_engineering.dataset_builder: Joined dataset: 1528 rows (2018-01-02 – 2024-03-04)


[Model B (FinBERT-LSTM) h5] start | window=10 horizon=5 n_iter=20 device=cuda
[Model B (FinBERT-LSTM) h5] split rows  train=1066 val=228 test=229
[Model B (FinBERT-LSTM) h5] split samples train=1057 val=219 test=220
[Model B (FinBERT-LSTM) h5] iter  1/20 val_f1=0.6886 val_acc=0.5251  <- NEW BEST
[Model B (FinBERT-LSTM) h5] iter  2/20 val_f1=0.6886 val_acc=0.5251
[Model B (FinBERT-LSTM) h5] iter  3/20 val_f1=0.6907 val_acc=0.5297  <- NEW BEST
[Model B (FinBERT-LSTM) h5] iter  4/20 val_f1=0.6886 val_acc=0.5251
[Model B (FinBERT-LSTM) h5] iter  5/20 val_f1=0.6886 val_acc=0.5251
[Model B (FinBERT-LSTM) h5] iter  6/20 val_f1=0.1860 val_acc=0.5205
[Model B (FinBERT-LSTM) h5] iter  7/20 val_f1=0.6886 val_acc=0.5251
[Model B (FinBERT-LSTM) h5] iter  8/20 val_f1=0.6975 val_acc=0.5525  <- NEW BEST
[Model B (FinBERT-LSTM) h5] iter  9/20 val_f1=0.6928 val_acc=0.5342
[Model B (FinBERT-LSTM) h5] iter 10/20 val_f1=0.6886 val_acc=0.5251
[Model B (FinBERT-LSTM) h5] iter 11/20 val_f1=0.6886 val_acc=0.52

## Bagian 4 — Perbandingan Horizon 1 (produksi) vs Horizon 5 (notebook ini)

Angka horizon-1 dibaca LANGSUNG dari metadata artefak produksi yang sudah tersimpan
(`lstm_baseline_skripsi_metadata.json` / `finbert_lstm_skripsi_metadata.json`) --
bukan angka hardcode -- supaya selalu sinkron dengan model produksi TERKINI (Random
Search bersifat stokastik, angka bisa berubah kalau model produksi dilatih ulang).

**Semua baris ditampilkan apa adanya, termasuk kalau horizon 5 ternyata TIDAK lebih
baik dari horizon 1 -- itu tetap temuan valid, bukan alasan untuk dicoba-coba lagi.**


In [7]:
def load_metadata(filename: str) -> dict:
    path = Path(ARTIFACT_DIR) / filename
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


meta_a_h1 = load_metadata(BASELINE_FILES["metadata"])
meta_b_h1 = load_metadata(FINBERT_FILES["metadata"])
meta_a_h5 = load_metadata(H5_BASELINE_FILES["metadata"])
meta_b_h5 = load_metadata(H5_FINBERT_FILES["metadata"])

rows = []
for label, meta in [
    ("Model A (OHLCV) -- horizon 1 (produksi)", meta_a_h1),
    ("Model A (OHLCV) -- horizon 5", meta_a_h5),
    ("Model B (FinBERT-LSTM) -- horizon 1 (produksi)", meta_b_h1),
    ("Model B (FinBERT-LSTM) -- horizon 5", meta_b_h5),
]:
    if not meta:
        rows.append({"Model": label, "Accuracy": None, "Precision": None, "Recall": None,
                      "F1": None, "TN": None, "FP": None, "FN": None, "TP": None})
        continue
    m, cm = meta["metrics"], meta["confusion_matrix"]
    rows.append({
        "Model": label,
        "Accuracy": m["accuracy"], "Precision": m["precision"],
        "Recall": m["recall"], "F1": m["f1"],
        "TN": cm["tn"], "FP": cm["fp"], "FN": cm["fn"], "TP": cm["tp"],
    })

comparison_df = pd.DataFrame(rows)
print("=== Perbandingan Horizon 1 vs Horizon 5 (dibaca dari metadata artefak) ===")
print(comparison_df.to_string(index=False))


=== Perbandingan Horizon 1 vs Horizon 5 (dibaca dari metadata artefak) ===
                                         Model  Accuracy  Precision   Recall       F1  TN  FP  FN  TP
       Model A (OHLCV) -- horizon 1 (produksi)  0.560000   0.560000 1.000000 0.717949   0  99   0 126
                  Model A (OHLCV) -- horizon 5  0.383929   0.681818 0.102740 0.178571  71   7 131  15
Model B (FinBERT-LSTM) -- horizon 1 (produksi)  0.438914   0.000000 0.000000 0.000000  97   0 124   0
           Model B (FinBERT-LSTM) -- horizon 5  0.404545   0.772727 0.118881 0.206061  72   5 126  17


## Catatan Penutup

- Horizon 1 tetap **satu-satunya hasil resmi** yang dilaporkan di BAB IV/V skripsi
  (Batasan Masalah BAB I mengunci horizon 1 hari).
- Horizon 5 di notebook ini adalah **analisis sensitivitas tambahan**, dilaporkan
  berdampingan tanpa menyembunyikan hasil yang kurang bagus.
- Kalau horizon 5 menang tipis di satu metrik tapi kalah di metrik lain (atau
  sebaliknya), JANGAN pilih horizon berdasarkan metrik mana yang kebetulan
  menguntungkan -- laporkan seluruh tabel apa adanya, sama seperti prinsip yang
  sudah dipegang di walk-forward/sensitivity analysis notebook 03.
- `notebook07_common.py` yang dimodifikasi (`horizon` param, default 1) sudah
  diverifikasi backward-compatible: 201/203 test lulus, 2 gagal adalah bug
  pre-existing yang tidak berkaitan (`test_default_ticker`,
  `test_sentiment_raw_analysis_endpoint[.../closing-price]`), tidak disebabkan
  perubahan ini.
